In [1]:
# IMPORTS

import os, gc, random, hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import torchvision.models as tv_models

import os
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
import mlflow
import mlflow.pytorch

from helper import AlexNetLike, plot_to_tensorboard, count_parameters

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device}')

c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.7'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


Usando: cpu


In [2]:
#DATASET

data_dir_total = r'data/Split_smol/'
valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

files_totales = []
for x in Path(data_dir_total).rglob('*'):
    if x.is_file() and x.suffix.lower() in valid_extensions:
        try:
            with Image.open(x) as img:
                files_totales.append((x, x.parent.name, img.size, img.mode))
        except Exception:
            pass

df_completo = pd.DataFrame(files_totales, columns=["path", "class", "resolution", "mode"])

def calcular_md5(path_objeto):
    hash_md5 = hashlib.md5()
    with open(path_objeto, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

df_completo['md5'] = df_completo['path'].apply(calcular_md5)
fotos_a_eliminar = {"aug_0_F2.large.jpg"}
df_completo = df_completo[~df_completo['path'].apply(lambda p: p.name).isin(fotos_a_eliminar)].reset_index(drop=True)
df_limpio = df_completo.drop_duplicates(subset=['md5'], keep='first').reset_index(drop=True)

df_train_val, df_test = train_test_split(df_limpio, test_size=0.20, stratify=df_limpio['class'], random_state=42)
df_train, df_val     = train_test_split(df_train_val, test_size=0.25, stratify=df_train_val['class'], random_state=42)

base_train_paths = df_train.reset_index(drop=True)["path"].apply(str).tolist()
val_image_paths  = df_val.reset_index(drop=True)["path"].apply(str).tolist()
test_image_paths = df_test.reset_index(drop=True)["path"].apply(str).tolist()

# Oversampling en train
random.seed(42)
train_image_paths = list(base_train_paths)
counts = Counter([Path(p).parent.name for p in train_image_paths])
max_count = max(counts.values())
for cls, count in counts.items():
    faltantes = max_count - count
    if faltantes > 0:
        cls_paths = [p for p in train_image_paths if Path(p).parent.name == cls]
        train_image_paths.extend(random.choices(cls_paths, k=faltantes))

print(f"Train: {len(train_image_paths)} | Val: {len(val_image_paths)} | Test: {len(test_image_paths)}")

Train: 540 | Val: 169 | Test: 169


In [3]:
class CustomImageDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform
        self.classes = sorted(set([Path(p).parent.name for p in image_paths]))
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.labels = [self.class_to_idx[Path(p).parent.name] for p in image_paths]

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image=image)["image"]
        return image, label

In [4]:
# FUNCION DE TRAIN

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(loader, desc="Train", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            out = model(images)
            total_loss += criterion(out, labels).item()
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total

def run_experiment(cfg):
    torch.manual_seed(cfg['seed'])
    np.random.seed(cfg['seed'])

    MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    size = cfg['input_size']

    aug_list = [A.Resize(size, size)]
    if cfg.get('hflip'):     aug_list.append(A.HorizontalFlip(p=cfg['hflip']))
    if cfg.get('vflip'):     aug_list.append(A.VerticalFlip(p=cfg['vflip']))
    if cfg.get('rbc'):       aug_list.append(A.RandomBrightnessContrast(p=cfg['rbc']))
    if cfg.get('clahe'):     aug_list.append(A.CLAHE(p=cfg['clahe']))
    if cfg.get('hsv'):       aug_list.append(A.HueSaturationValue(p=cfg['hsv']))
    if cfg.get('rotate'):    aug_list.append(A.Rotate(limit=20, p=cfg['rotate']))
    aug_list += [A.Normalize(mean=MEAN, std=STD), ToTensorV2()]

    train_tf = A.Compose(aug_list)
    val_tf   = A.Compose([A.Resize(size, size), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])

    train_ds = CustomImageDataset(train_image_paths, transform=train_tf)
    val_ds   = CustomImageDataset(val_image_paths,   transform=val_tf)
    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

    model = AlexNetLike(input_size=size, dropout=cfg['dropout'], num_classes=9).to(device)
    print(f"Parámetros: {count_parameters(model):,}")

    criterion = nn.CrossEntropyLoss()
    if cfg['optimizer'] == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=cfg['lr'], momentum=cfg['momentum'], weight_decay=cfg['weight_decay'])
    else:
        optimizer = optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])

    mlflow.set_experiment("CNN_Dermatologia")
    best_val_acc, best_path = 0, f"best_{cfg['run_name']}.pth"
    epochs_sin_mejora = 0
    writer = SummaryWriter(log_dir=f"runs/{cfg['run_name']}")

    with mlflow.start_run(run_name=cfg['run_name']):
        mlflow.log_params(cfg)

        for epoch in range(cfg['n_epochs']):
            t_loss, t_acc = train_epoch(model, train_loader, optimizer, criterion)
            v_loss, v_acc = evaluate(model, val_loader, criterion)

            print(f"Epoch {epoch+1:3d} | Train: {t_acc:.2f}% | Val: {v_acc:.2f}%")
            writer.add_scalars("Loss",     {"train": t_loss, "val": v_loss}, epoch)
            writer.add_scalars("Accuracy", {"train": t_acc,  "val": v_acc},  epoch)
            mlflow.log_metrics({"train_loss": t_loss, "train_acc": t_acc, "val_loss": v_loss, "val_acc": v_acc}, step=epoch)

            if v_acc > best_val_acc:
                best_val_acc = v_acc
                epochs_sin_mejora = 0
                torch.save(model.state_dict(), best_path)
            else:
                epochs_sin_mejora += 1
                if epochs_sin_mejora >= cfg['es_patience']:
                    print(f"Early stopping en epoch {epoch+1}")
                    break

        mlflow.log_metric("best_val_acc", best_val_acc)

    writer.close()
    print(f"\n Mejor Val Acc: {best_val_acc:.2f}%")
    return model, best_val_acc, best_path

In [5]:
# EXPERIMENTOS

CFG = {
    'run_name': 'campeon_rs040',
    'input_size': 64,
    'dropout':    0.3,
    'optimizer':  'Adam',
    'lr':         1e-4,
    'momentum':   0.99,
    'weight_decay': 0,
    'batch_size': 16,
    'n_epochs':   60,
    'es_patience': 7,
    'seed':       42,
    'hflip':  0.0,
    'vflip':  0.5,
    'rbc':    0.5,
    'clahe':  0.3,
    'hsv':    0.0,
    'rotate': 0.4,
}

In [ ]:
model, best_val_acc, best_path = run_experiment(CFG)
print(f"Run: {CFG['run_name']} → Val Acc: {best_val_acc:.2f}%")

2026/06/12 15:45:43 INFO mlflow.tracking.fluent: Experiment with name 'CNN_Dermatologia' does not exist. Creating a new experiment.


Parámetros: 2,546,761


Train:   0%|          | 0/34 [00:00<?, ?it/s]c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
## BONUS : Transfer Learning con ResNet18

INPUT_SIZE_TL = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf_tl = A.Compose([
    A.Resize(INPUT_SIZE_TL, INPUT_SIZE_TL),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5), A.CLAHE(p=0.3),
    A.HueSaturationValue(p=0.3), A.Rotate(limit=20, p=0.4),
    A.Normalize(mean=MEAN, std=STD), ToTensorV2()
])
val_tf_tl = A.Compose([A.Resize(INPUT_SIZE_TL, INPUT_SIZE_TL), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])

train_ds_tl = CustomImageDataset(train_image_paths, transform=train_tf_tl)
val_ds_tl   = CustomImageDataset(val_image_paths,   transform=val_tf_tl)
train_loader_tl = DataLoader(train_ds_tl, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader_tl   = DataLoader(val_ds_tl,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

classes_tl = train_ds_tl.classes
NUM_CLASSES = len(classes_tl)
print(f"Clases: {classes_tl}")

In [ ]:
#  Feature Extraction 
resnet = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)

for param in resnet.parameters():
    param.requires_grad = False

in_features = resnet.fc.in_features
resnet.fc = nn.Sequential(
    nn.Linear(in_features, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, NUM_CLASSES)
)
resnet = resnet.to(device)
print(f"Parámetros entrenables fase 1: {count_parameters(resnet):,}")

In [ ]:
criterion_tl = nn.CrossEntropyLoss()
optimizer_f1 = optim.Adam(filter(lambda p: p.requires_grad, resnet.parameters()), lr=1e-3)

best_val_tl, path_tl = 0, "best_resnet18_fase1.pth"
mlflow.set_experiment("CNN_Dermatologia")

with mlflow.start_run(run_name="tl_resnet18_fase1"):
    mlflow.log_params({"model": "ResNet18", "fase": "feature_extraction", "lr": 1e-3, "batch": 32, "input": 224})
    for epoch in range(20):
        t_loss, t_acc = train_epoch(resnet, train_loader_tl, optimizer_f1, criterion_tl)
        v_loss, v_acc = evaluate(resnet, val_loader_tl, criterion_tl)
        print(f"Epoch {epoch+1:2d} | Train: {t_acc:.2f}% | Val: {v_acc:.2f}%")
        mlflow.log_metrics({"train_acc": t_acc, "val_acc": v_acc}, step=epoch)
        if v_acc > best_val_tl:
            best_val_tl = v_acc
            torch.save(resnet.state_dict(), path_tl)
    mlflow.log_metric("best_val_acc_fase1", best_val_tl)

print(f"\nFase 1 terminada. Mejor Val Acc: {best_val_tl:.2f}%")

In [ ]:
# Fine-tuning 
resnet.load_state_dict(torch.load(path_tl))

for name, param in resnet.named_parameters():
    param.requires_grad = any(l in name for l in ['layer3', 'layer4', 'fc'])

print(f"Parámetros entrenables fase 2: {count_parameters(resnet):,}")

optimizer_f2 = optim.SGD(filter(lambda p: p.requires_grad, resnet.parameters()), lr=1e-4, momentum=0.9, weight_decay=1e-4)

best_val_ft, path_ft = best_val_tl, "best_resnet18_finetune.pth"
epochs_sin_mejora = 0

with mlflow.start_run(run_name="tl_resnet18_fase2_finetune"):
    mlflow.log_params({"model": "ResNet18", "fase": "fine_tuning", "lr": 1e-4, "capas": "layer3+layer4+fc"})
    for epoch in range(40):
        t_loss, t_acc = train_epoch(resnet, train_loader_tl, optimizer_f2, criterion_tl)
        v_loss, v_acc = evaluate(resnet, val_loader_tl, criterion_tl)
        print(f"Epoch {epoch+1:2d} | Train: {t_acc:.2f}% | Val: {v_acc:.2f}%")
        mlflow.log_metrics({"train_acc": t_acc, "val_acc": v_acc}, step=epoch)
        if v_acc > best_val_ft:
            best_val_ft = v_acc
            epochs_sin_mejora = 0
            torch.save(resnet.state_dict(), path_ft)
        else:
            epochs_sin_mejora += 1
            if epochs_sin_mejora >= 7:
                print(f"Early stopping en epoch {epoch+1}")
                break
    mlflow.log_metric("best_val_acc_fase2", best_val_ft)

print(f"\nFine-tuning terminado. Mejor Val Acc: {best_val_ft:.2f}%")

In [ ]:
# TEST: VA COMENTADO HASTA EL FINAL, SE CORRE UNA SOLA VEZ

CAMPEÓN_PATH = path_tl
CAMPEÓN_ES_RESNET = True
CAMPEÓN_INPUT_SIZE = 224

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
test_tf = A.Compose([A.Resize(CAMPEÓN_INPUT_SIZE, CAMPEÓN_INPUT_SIZE), A.Normalize(mean=MEAN, std=STD), ToTensorV2()])
test_ds = CustomImageDataset(test_image_paths, transform=test_tf)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

if CAMPEÓN_ES_RESNET:
    campeon = tv_models.resnet18(weights=None)
    campeon.fc = nn.Sequential(nn.Linear(campeon.fc.in_features, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, 9))
else:
    campeon = AlexNetLike(input_size=CAMPEÓN_INPUT_SIZE, dropout=0.3, num_classes=9)

campeon.load_state_dict(torch.load(CAMPEÓN_PATH))
campeon = campeon.to(device)

test_loss, test_acc = evaluate(campeon, test_loader, nn.CrossEntropyLoss())
print(f"TEST Accuracy: {test_acc:.2f}% | Loss: {test_loss:.4f}")

campeon.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        preds = campeon(images.to(device)).argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

classes_test = test_ds.classes
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(cm, display_labels=classes_test).plot(ax=ax, cmap='Blues', xticks_rotation=45)
ax.set_title('TEST — Confusion Matrix — Modelo Campeón')
plt.tight_layout()
# 
# os.makedirs('imagenes', exist_ok=True)
plt.savefig('imagenes/test_confusion_matrix.png', dpi=150)
plt.show()
print(classification_report(all_labels, all_preds, target_names=classes_test, zero_division=0))

with mlflow.start_run(run_name='TEST_FINAL'):
    mlflow.log_metrics({'test_accuracy': test_acc, 'test_loss': test_loss})
    mlflow.log_artifact('imagenes/test_confusion_matrix.png')